# Probar modelo

In [1]:
import torch
import timm
import os
from PIL import Image
from torchvision import transforms
import json
import torch.nn.functional as F

D:\entornos\tesis\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
D:\entornos\tesis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Cargar modelo entrenado

In [3]:
carpeta_modelos = "modelos"
os.makedirs(carpeta_modelos, exist_ok=True)

with open("clases.json", "r") as f:
    clases = json.load(f)
    
num_clases = len(clases)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

modelo_vit = timm.create_model("vit_tiny_patch16_224", pretrained=False, num_classes=num_clases)
modelo_vit.load_state_dict(
    torch.load(os.path.join(carpeta_modelos, "modelo_vit_final.pth"), map_location=device)
)
modelo_vit = modelo_vit.to(device)
modelo_vit.eval()

C:\Users\Generacion 4\AppData\Local\Temp\ipykernel_9696\295055849.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(carpeta_modelos, "modelo_vit_f

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=192, out_features=576, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=192, out_features=192, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=192, out_features=768, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)


# Transformaciones para la imagen

In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean= [0.485, 0.456, 0.406],  # Igual que en entrenamiento
                         std= [0.229, 0.224, 0.225])
])

# Función para predecir

In [5]:
def predecir_imagen(ruta_imagen):
    imagen = Image.open(ruta_imagen).convert("RGB")
    imagen_tensor = transform(imagen).unsqueeze(0).to(device)

    with torch.no_grad():
        salida = modelo_vit(imagen_tensor)
        _, pred = torch.max(salida, 1)

    clase_predicha = clases[pred.item()]
    return clase_predicha

In [6]:
# Ejemplo de uso

ruta = r"E:\Mis Documentos\Christian\Tesis\Test\test2.jpg"
ruta2 = r"E:\Mis Documentos\Christian\Tesis\Test\test5.jpg"
print("Predicción:", predecir_imagen(ruta))

print("Predicción 2:", predecir_imagen(ruta2))

Predicción: 2
Predicción 2: 0
